In [1]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv('../data/training_data.csv')
print(df.shape)
print(df.head())

(3017586, 12)
   p1_rating_before  p1_rank  p1_power  p1_overall_wr  p1_char_wr  \
0              1338       23    190164       0.346154    0.346154   
1              1918       27    365620       0.557143    0.557143   
2              1326       16    120528       0.476190    0.476190   
3              1505       24    197332       0.592593    0.592593   
4              1469       23    196946       0.482143    0.575758   

   p2_rating_before  p2_rank  p2_power  p2_overall_wr  p2_char_wr  matchup_wr  \
0              1399       23    191175       0.333333    0.333333    0.494962   
1              1660       26    237387       0.000000    0.000000    0.476821   
2              1265       15    145756       0.272727    0.333333    0.495907   
3              1504       24    232311       0.500000    0.500000    0.540708   
4              1414       23    203489       0.466667    0.259259    0.498515   

   winner  
0       1  
1       1  
2       1  
3       2  
4       1  


In [3]:
p1_cols = ['p1_rating_before', 'p1_rank', 'p1_power', 'p1_overall_wr', 'p1_char_wr']
p2_cols = ['p2_rating_before', 'p2_rank', 'p2_power', 'p2_overall_wr', 'p2_char_wr']

X_p1 = df[p1_cols].values
X_p2 = df[p2_cols].values
X_matchup = df[['matchup_wr']].values
y = (df['winner'] - 1).values

print('X_p1:', X_p1.shape)
print('X_p2:', X_p2.shape)
print('X_matchup:', X_matchup.shape)
print('y:', y.shape)

(3017586, 11)
(3017586,)


In [4]:
X_p1_train, X_p1_val, X_p2_train, X_p2_val, X_matchup_train, X_matchup_val, y_train, y_val = train_test_split(
    X_p1, X_p2, X_matchup, y, test_size=0.2, random_state=42
)
print('Train:', X_p1_train.shape, X_p2_train.shape, X_matchup_train.shape)
print('Val:  ', X_p1_val.shape, X_p2_val.shape, X_matchup_val.shape)

(2414068, 11)
(603518, 11)


In [5]:
X_p1_train = torch.tensor(X_p1_train, dtype=torch.float32)
X_p1_val = torch.tensor(X_p1_val, dtype=torch.float32)
X_p2_train = torch.tensor(X_p2_train, dtype=torch.float32)
X_p2_val = torch.tensor(X_p2_val, dtype=torch.float32)
X_matchup_train = torch.tensor(X_matchup_train, dtype=torch.float32)
X_matchup_val = torch.tensor(X_matchup_val, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.float32)

In [6]:
class PlayerEncoder(nn.Module):
    """Layer 1: encodes one player's 5 features into a 32-dim embedding."""
    def __init__(self, input_dim=5, hidden_dim=64, embed_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embed_dim),
        )

    def forward(self, x):
        return self.encoder(x)

player_encoder = PlayerEncoder()
print(player_encoder)

# Quick sanity check: same weights used for both players
embed_p1 = player_encoder(X_p1_train[:4])
embed_p2 = player_encoder(X_p2_train[:4])
print('P1 embedding shape:', embed_p1.shape)
print('P2 embedding shape:', embed_p2.shape)

TekkenEmbedding(
  (embedding): Sequential(
    (0): Linear(in_features=11, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
  )
)


In [ ]:
class MatchPredictor(nn.Module):
    """Full model: Layer 1 (PlayerEncoder) + Layer 2 (binary classifier)."""
    def __init__(self, player_input_dim=5, hidden_dim=64, embed_dim=32, classifier_hidden=32):
        super().__init__()
        self.player_encoder = PlayerEncoder(player_input_dim, hidden_dim, embed_dim)
        combined_dim = embed_dim * 2 + 1  # both embeddings + matchup_wr
        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, classifier_hidden),
            nn.ReLU(),
            nn.Linear(classifier_hidden, 1),
            nn.Sigmoid(),
        )

    def forward(self, p1_features, p2_features, matchup_wr):
        embed_p1 = self.player_encoder(p1_features)
        embed_p2 = self.player_encoder(p2_features)
        combined = torch.cat([embed_p1, embed_p2, matchup_wr], dim=1)
        return self.classifier(combined)

model = MatchPredictor()
print(model)

with torch.no_grad():
    sample_probs = model(X_p1_train[:4], X_p2_train[:4], X_matchup_train[:4])
print('Output shape:', sample_probs.shape)
print('Sample probabilities:', sample_probs.squeeze().tolist())

In [ ]:
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
EPOCHS = 5
BATCH_SIZE = 1024

train_dataset = TensorDataset(
    X_p1_train, X_p2_train, X_matchup_train, y_train.unsqueeze(1)
)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

model.train()
for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for p1_batch, p2_batch, matchup_batch, y_batch in train_loader:
        optimizer.zero_grad()
        predictions = model(p1_batch, p2_batch, matchup_batch)
        loss = criterion(predictions, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    print(f'Epoch {epoch + 1}/{EPOCHS} — train loss: {avg_loss:.4f}')

In [ ]:
model.eval()
with torch.no_grad():
    val_predictions = model(X_p1_val, X_p2_val, X_matchup_val)
    val_loss = criterion(val_predictions, y_val.unsqueeze(1))
    val_accuracy = ((val_predictions > 0.5) == y_val.unsqueeze(1)).float().mean()

print(f'Validation loss: {val_loss.item():.4f}')
print(f'Validation accuracy: {val_accuracy.item():.4f} ({val_accuracy.item() * 100:.1f}%)')

In [ ]:
from pathlib import Path

save_path = Path('../models/match_predictor.pt')
save_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), save_path)
print(f'Model saved to {save_path.resolve()}')